In [1]:
import torch
import torch.nn as nn

from typing import List
# import tqdm
from tqdm import tqdm


In [2]:
class BaseTokenizer:
    unk_token = "<unk>"
    pad_token = "<pad>"
    sos_token = "<s>"
    eos_token = "</s>"

    def __init__(self, vocab_list):
        """外部的调用会无法正常初始化<pad>和<unk>, 当前仅供内部调用"""
        self.vocab_list = vocab_list
        self.vocab_size = len(self.vocab_list)
        self.index2word = self.vocab_list
        self.word2index = {word: index for index, word in enumerate(self.index2word)}
        self.unk_token_index = self.word2index[self.unk_token]
        self.pad_token_index = self.word2index[self.pad_token]
        self.sos_token_index = self.word2index[self.sos_token]
        self.eos_token_index = self.word2index[self.eos_token]

    @staticmethod
    def tokenize(text) -> List[str]:
        return list(text)

    def encode(self, text):
        tokens = self.tokenize(text)
        return [self.word2index.get(token, self.unk_token_index) for token in tokens]
    
    def decode(self, indexes) -> str:
        return [self.index2word[index] for index in indexes]

    @classmethod
    def build_vocab(cls, texts):
        vocab_set = set()
        for text in tqdm(texts, desc="Building vocab"):
            tokens = cls.tokenize(text)
            vocab_set.update(tokens)

        vocab_list = [cls.pad_token, cls.unk_token, cls.sos_token, cls.eos_token] + [token for token in vocab_set if token and token.strip()]

        return cls(vocab_list)

In [3]:
text = "人生得意须尽欢，莫使金樽空对月"

tokenizer = BaseTokenizer.build_vocab([text])


Building vocab: 100%|██████████| 1/1 [00:00<?, ?it/s]


In [7]:
enc_tokens = []
dec_tokens = []

for i in range(1,len(text)):
        enc = text[:i]
        dec = text[i:]
        enc_tokens.append(enc)
        dec_tokens.append(dec)

In [8]:
enc_tokens

['人',
 '人生',
 '人生得',
 '人生得意',
 '人生得意须',
 '人生得意须尽',
 '人生得意须尽欢',
 '人生得意须尽欢，',
 '人生得意须尽欢，莫',
 '人生得意须尽欢，莫使',
 '人生得意须尽欢，莫使金',
 '人生得意须尽欢，莫使金樽',
 '人生得意须尽欢，莫使金樽空',
 '人生得意须尽欢，莫使金樽空对']

In [9]:
dec_tokens

['生得意须尽欢，莫使金樽空对月',
 '得意须尽欢，莫使金樽空对月',
 '意须尽欢，莫使金樽空对月',
 '须尽欢，莫使金樽空对月',
 '尽欢，莫使金樽空对月',
 '欢，莫使金樽空对月',
 '，莫使金樽空对月',
 '莫使金樽空对月',
 '使金樽空对月',
 '金樽空对月',
 '樽空对月',
 '空对月',
 '对月',
 '月']

In [13]:
enc_input_list = []
dec_input_list = []
dec_target_list = []
for enc_data, dec_data in zip(enc_tokens, dec_tokens):
    enc_inputs = tokenizer.encode(enc_data)
    dec_encoded_data = tokenizer.encode(dec_data)
    dec_inputs = [tokenizer.sos_token_index] + dec_encoded_data
    dec_targets = dec_encoded_data + [tokenizer.eos_token_index]
    enc_input_list.append(enc_inputs)
    dec_input_list.append(dec_inputs)
    dec_target_list.append(dec_targets)

In [26]:
for a, b, c in zip(enc_input_list, dec_input_list, dec_target_list):
    print(a, b, c)


[8] [2, 14, 15, 17, 13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11] [14, 15, 17, 13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11, 3]
[8, 14] [2, 15, 17, 13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11] [15, 17, 13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11, 3]
[8, 14, 15] [2, 17, 13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11] [17, 13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11, 3]
[8, 14, 15, 17] [2, 13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11] [13, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11, 3]
[8, 14, 15, 17, 13] [2, 16, 5, 9, 10, 4, 7, 18, 6, 12, 11] [16, 5, 9, 10, 4, 7, 18, 6, 12, 11, 3]
[8, 14, 15, 17, 13, 16] [2, 5, 9, 10, 4, 7, 18, 6, 12, 11] [5, 9, 10, 4, 7, 18, 6, 12, 11, 3]
[8, 14, 15, 17, 13, 16, 5] [2, 9, 10, 4, 7, 18, 6, 12, 11] [9, 10, 4, 7, 18, 6, 12, 11, 3]
[8, 14, 15, 17, 13, 16, 5, 9] [2, 10, 4, 7, 18, 6, 12, 11] [10, 4, 7, 18, 6, 12, 11, 3]
[8, 14, 15, 17, 13, 16, 5, 9, 10] [2, 4, 7, 18, 6, 12, 11] [4, 7, 18, 6, 12, 11, 3]
[8, 14, 15, 17, 13, 16, 5, 9, 10, 4] [2, 7, 18, 6, 12, 11] [7, 18, 6, 12, 11, 3]
[8, 14, 15, 17, 13, 16, 5, 9, 10

### 处理数据

In [51]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def get_proc(batch):
    enc_input_list, dec_input_list, dec_target_list = zip(*batch)
    enc_input_list = [torch.tensor(x) for x in enc_input_list]
    dec_input_list = [torch.tensor(x) for x in dec_input_list]
    dec_target_list = [torch.tensor(x) for x in dec_target_list]
    enc_input_list = pad_sequence(enc_input_list, batch_first=True, padding_value=0)
    dec_input_list = pad_sequence(dec_input_list, batch_first=True, padding_value=0)
    dec_target_list = pad_sequence(dec_target_list, batch_first=True, padding_value=0)
    return enc_input_list, dec_input_list, dec_target_list

dl = DataLoader(
    list(zip(enc_input_list, dec_input_list, dec_target_list)),
    batch_size=256,
    shuffle=True,
    collate_fn=get_proc
)

for batch in dl:
    print(batch)

(tensor([[ 8, 14, 15, 17, 13,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7, 18,  6,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7, 18,  6, 12],
        [ 8, 14, 15, 17, 13, 16,  5,  0,  0,  0,  0,  0,  0,  0],
        [ 8,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7, 18,  0,  0],
        [ 8, 14,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]]), tensor([[ 2, 16,  5,  9, 10,  4,  7, 18,  6, 12, 11,  0,  0,  0,  0],
   

## 模型定义

In [68]:
import math

# 位置编码矩阵
class PositionalEncoding(nn.Module):

    def __init__(self, emb_size, dropout, maxlen=5000):
        super().__init__()
        # 行缩放指数值
        den = torch.exp(- torch.arange(0, emb_size, 2) * math.log(10000) / emb_size)
        # 位置编码索引 (5000,1)
        pos = torch.arange(0, maxlen).reshape(maxlen, 1)
        # 编码矩阵 (5000, emb_size)
        pos_embdding = torch.zeros((maxlen, emb_size))
        pos_embdding[:, 0::2] = torch.sin(pos * den)
        pos_embdding[:, 1::2] = torch.cos(pos * den)
        # 添加和batch对应维度 (1, 5000, emb_size)
        # pos_embdding = pos_embdding.unsqueeze(-2)
        # dropout
        self.dropout = nn.Dropout(dropout)
        # 注册当前矩阵不参与参数更新
        self.register_buffer('pos_embedding', pos_embdding)

    def forward(self, token_embdding):
        token_len = token_embdding.size(1)  # token长度
        # (token_len, emb_size)
        add_emb = self.pos_embedding[:token_len, :] + token_embdding
        return self.dropout(add_emb)

class Seq2SeqTransformer(nn.Module):

    def __init__(self, d_model, nhead, num_enc_layers, num_dec_layers, 
                 dim_forward, dropout, enc_voc_size, dec_voc_size):
        super().__init__()
        # transformer
        self.transformer = nn.Transformer(d_model=d_model,
                                          nhead=nhead,
                                          num_encoder_layers=num_enc_layers,
                                          num_decoder_layers=num_dec_layers,
                                          dim_feedforward=dim_forward,
                                          dropout=dropout,
                                          batch_first=True)
        # encoder input embedding
        self.enc_emb = nn.Embedding(enc_voc_size, d_model)
        # decoder input embedding
        self.dec_emb = nn.Embedding(dec_voc_size, d_model)
        # predict generate linear
        self.predict = nn.Linear(d_model, dec_voc_size)  # token预测基于解码器词典
        # positional encoding
        self.pos_encoding = PositionalEncoding(d_model, dropout)

    def forward(self, enc_inp, dec_inp, tgt_mask=None, enc_pad_mask=None, dec_pad_mask=None):
        if tgt_mask is None:
            tgt_mask = self.transformer.generate_square_subsequent_mask(dec_inp.size(1)).to(dec_inp.device)
        if enc_pad_mask is None:
            enc_pad_mask = enc_inp == 0
        if dec_pad_mask is None:
            dec_pad_mask = dec_inp == 0
        # multi head attention之前基于位置编码embedding生成
        enc_emb = self.pos_encoding(self.enc_emb(enc_inp))
        dec_emb = self.pos_encoding(self.dec_emb(dec_inp))
        # 调用transformer计算
        outs = self.transformer(src=enc_emb, tgt=dec_emb, tgt_mask=tgt_mask,
                         src_key_padding_mask=enc_pad_mask, 
                         tgt_key_padding_mask=dec_pad_mask)
        # 推理
        return self.predict(outs)
    
    # 推理环节使用方法
    def encode(self, enc_inp, enc_pad_mask=None):
        if enc_pad_mask is None:
            enc_pad_mask = enc_inp == 0
        enc_emb = self.pos_encoding(self.enc_emb(enc_inp))
        return self.transformer.encoder(enc_emb, src_key_padding_mask=enc_pad_mask)
    
    def decode(self, dec_inp, memory, memory_pad_mask, dec_mask=None, dec_key_padding_mask=None):
        if dec_mask is None:
            dec_mask = self.transformer.generate_square_subsequent_mask(dec_inp.size(1)).to(dec_inp.device)
        if dec_key_padding_mask is None:
            dec_key_padding_mask = dec_inp == 0
        dec_emb = self.pos_encoding(self.dec_emb(dec_inp))
        output = self.transformer.decoder(dec_emb, memory, 
                                        tgt_mask=dec_mask, 
                                        memory_key_padding_mask=memory_pad_mask,
                                        tgt_key_padding_mask=dec_key_padding_mask)
        return self.predict(output)


In [49]:
for item in dl:
    print(item[0])

(tensor([[ 8, 14, 15, 17, 13, 16,  5,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7, 18,  6,  0],
        [ 8, 14, 15,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  0,  0,  0,  0],
        [ 8,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9,  0,  0,  0,  0,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7, 18,  6, 12],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  4,  7, 18,  0,  0],
        [ 8, 14, 15, 17, 13, 16,  5,  9, 10,  0,  0,  0,  0,  0]]),)


## 开始训练

In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LEARNING_RATE = 0.001

model = Seq2SeqTransformer(d_model=128, 
                           nhead=2, 
                           num_enc_layers=2, 
                           num_dec_layers=2, 
                           dim_forward=256, 
                           dropout=0.1, 
                           enc_voc_size=tokenizer.vocab_size, 
                           dec_voc_size=tokenizer.vocab_size).to(device)
# 损失函数
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_index)
# 优化器
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


min_loss = float('inf')
model.train()
for epoch in tqdm(range(10), desc='Training'):
    for batch in dl:
        src, tgt_in, tgt_out = batch
        
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)
        # 前向传播
        output = model(src, tgt_in)
        # 计算损失
        loss = loss_fn(output.view(-1, output.size(-1)), tgt_out.view(-1))
        if loss.item() < min_loss:
            min_loss = loss.item()
            torch.save(model.state_dict(), 'best_model.pth')
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


Training:   0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\MI\miniconda3\envs\nlp\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Training: 100%|██████████| 10/10 [00:00<00:00, 17.34it/s]


In [46]:
a,b,c= (1, 2, 3)

type(a)

int

## 开始预测

In [71]:
model = Seq2SeqTransformer(d_model=128, 
                           nhead=2, 
                           num_enc_layers=2, 
                           num_dec_layers=2, 
                           dim_forward=256, 
                           dropout=0.1, 
                           enc_voc_size=tokenizer.vocab_size, 
                           dec_voc_size=tokenizer.vocab_size).to(device)
model.load_state_dict(torch.load('best_model.pth'))

print("模型加载成功")

MAX_SEQ_LENGTH = 128

def predict_batch(model, input_ids):
    model.eval()
    with torch.no_grad():
        memory = model.encode(input_ids)
        src_pad_mask = input_ids == 0
        batch_size = input_ids.size(0)
        decoder_input = torch.full((batch_size, 1), 
                                   tokenizer.sos_token_index, 
                                   dtype=torch.long).to(device)
        generated_tokens = []

        is_finished = torch.full([batch_size], False, dtype=torch.bool).to(device)
        
        for _ in range(MAX_SEQ_LENGTH):
            decoder_output = model.decode(decoder_input, memory, memory_pad_mask=src_pad_mask)
            # decoder_output.shape: [batch_size, tgt_seq_len, en_vocab_size]
            next_token_indexes = torch.argmax(decoder_output[:, -1, :], dim=-1, keepdim=True)
            # next_token_indexes.shape: [batch_size, 1]
            generated_tokens.append(next_token_indexes)

            # 更新输入
            decoder_input = torch.cat([decoder_input, next_token_indexes], dim=1)

            # 判断是否应该结束
            is_finished |= (next_token_indexes.squeeze(1) == tokenizer.eos_token_index)

            if is_finished.all():
                break
        
        # 处理预测结果
        generated_tensor = torch.cat(generated_tokens, dim=1)
        # generated_tensor.shape: [batch_size, seq_len]
        # 移除结束标记
        generated_list = generated_tensor.tolist()
        for idx, sentence in enumerate(generated_list):
            if tokenizer.eos_token_index in sentence:
                generated_list[idx] = sentence[:sentence.index(tokenizer.eos_token_index)]
    return generated_list



def predict(text, model):
    input_ids = tokenizer.encode(text)
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

    batch_result = predict_batch(model, input_ids)

    return tokenizer.decode(batch_result[0])

def run_predict():
    while True:
        text = input("输入文本(输入q结束): ")
        if text == 'q':
            break

        res = predict(text, model)
        print(res)
if __name__ == '__main__':
    run_predict()

模型加载成功
['欢', '，', '莫', '使', '金', '樽', '空', '对', '月']
